In [ ]:
"""
Customer360 Navigator Enterprise Suite - BP4 Decision-Artifact Persistence notebook (Hardening
Step 2). Single consolidated code cell (platform convention, mirrors BP1/BP2's model-persistence
notebooks - see their module docstrings for the same structural rationale).

BP4 (Customer Journey Analytics) fits no supervised model - the user's own explicit standing
choice for BP4 hardening (confirmed via AskUserQuestion in an earlier session) is "Persist +
serve decision artifacts", not "persist a model". This notebook is BP4's structural analogue of
BP1/BP2/BP3's model-persistence step: it takes the real, already Gate-5-confirmed per-cluster
decision/reporting records CSV (notebooks/bp4_customer_journey_analytics/artifacts/
gate5_cluster_decision_report.csv - written by
bp4_customer_journey_analytics_g5_decision_layer_reporting.ipynb, one row per real issue-cluster,
BP4's own real, disclosed (Company, Product, Sub-product, Issue, Sub-issue) key, live-verified
unique with zero duplicates) and persists it into a fast-loadable, typed, sorted Parquet index
(models/bp4_customer_journey_analytics/bp4_decision_artifact_index.parquet) that Hardening Step 3's
read-only lookup/query FastAPI service loads at startup - never a trained model, never a fabricated
substitute for one.

No customer/consumer identifier exists anywhere in this real CFPB extract (BP4 Gate 1's own live-
verified finding, restated in every BP4 deliverable per the project's standing naming commitment).
The issue-cluster key (Company, Product, Sub-product, Issue, Sub-issue) is therefore both the only
real, disclosed identifier this artifact carries AND the natural physical sort/index key for a
lookup service's most common real query pattern - "look up this specific issue-cluster's review
priority" - so the persisted Parquet is written sorted by that key (never by a customer field,
which does not exist, and never a guessed column).

Run only after BP4 Gate 5 (Decision Layer & Reporting) is real-run confirmed (this notebook re-
derives that prerequisite live from configs/bp4_customer_journey_analytics.yaml, never trusts a
cached status string). Idempotent - safe to re-run; this notebook additionally self-checks
idempotency in-run (Section 8) by writing the Parquet artifact twice and asserting byte-identical
sha256 hashes, not merely asserting the file exists.

Reuses src/features/bp4_journey_features.py's own CLUSTER_KEY / BARRED_JOURNEY_COLUMNS constants
(Gate 2's own module, HYPER - never re-typed here a second time) and src/utils/bp1_config_sync.py's
write_gate_block() (generic, reused unmodified across every BP in this project) for the config
write.
"""

import os, sys, json, hashlib, warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")


# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
def _find_project_root() -> Path:
    marker = "PROJECT_STRUCTURE_LOCKED.md"
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        if (Path(env_override) / marker).exists():
            return Path(env_override)
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {env_override!r} but {marker} was not found there. "
            "Fix the environment variable rather than removing this check."
        )
    start = Path.cwd()
    cur = start
    for _ in range(8):
        if (cur / marker).exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker in filenames:
            return Path(depth_root)
    raise RuntimeError(
        f"Could not resolve PROJECT_ROOT: no {marker} found by walking up from {start}, nor by "
        "searching up to 3 levels below it. Fix: add a cell at the TOP of this notebook (before "
        "this cell runs) with:\n"
        '    import os; os.environ["C360_PROJECT_ROOT"] = r"C:\\Users\\rnand\\Documents\\'
        'Customer360_Navigator_Enterprise_Suite"\n'
        "then re-run from the top."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP performance configuration - FIRST, before any heavy import
# ============================================================
from utils.performance_setup import configure_performance  # noqa: E402

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)

# ============================================================
# SECTION 3: Heavy imports + flush-forcing print override
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402

import pandas as pd  # noqa: E402
import polars as pl  # noqa: E402
import yaml  # noqa: E402

print = functools.partial(builtins.print, flush=True)

from features.bp4_journey_features import BARRED_JOURNEY_COLUMNS, CLUSTER_KEY  # noqa: E402
from utils.bp1_config_sync import write_gate_block  # noqa: E402

CONFIGS_DIR = PROJECT_ROOT / "configs"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp4_customer_journey_analytics" / "artifacts"
MODELS_DIR = PROJECT_ROOT / "models" / "bp4_customer_journey_analytics"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
BP4_CONFIG_PATH = CONFIGS_DIR / "bp4_customer_journey_analytics.yaml"

for p in (CONFIGS_DIR, ARTIFACTS_DIR, BP4_CONFIG_PATH):
    if not p.exists():
        raise FileNotFoundError(f"Required path not found: {p}. Confirm BP4 Gates 1-5 completed for real.")

# ============================================================
# SECTION 4: Prerequisite check - BP4 Gate 5's real results, re-derived live from the config file
# (Three-Lines-of-Defense pattern already used by every gate in this project - never trusts a
# cached status string alone).
# ============================================================
with open(BP4_CONFIG_PATH, "r", encoding="utf-8") as f:
    FULL_CONFIG = yaml.safe_load(f)

gate5_confirmed = (
    isinstance(FULL_CONFIG.get("n_clusters_reported"), int)
    and FULL_CONFIG.get("n_clusters_reported", 0) > 0
    and isinstance(FULL_CONFIG.get("tier_rollup"), dict)
    and set(FULL_CONFIG["tier_rollup"].keys()) == {"HIGH", "MEDIUM", "LOW", "NONE"}
)
assert gate5_confirmed, (
    "BP4 Gate 5 (Decision Layer & Reporting) does not appear to have completed successfully "
    f"(n_clusters_reported={FULL_CONFIG.get('n_clusters_reported')!r}, "
    f"tier_rollup={FULL_CONFIG.get('tier_rollup')!r}). Run Gate 5 for real before this step."
)
N_CLUSTERS_EXPECTED = FULL_CONFIG["n_clusters_reported"]
print(
    f"[OK] BP4 Gate 5 prerequisite live-reconfirmed: n_clusters_reported={N_CLUSTERS_EXPECTED:,}, "
    f"tier_rollup keys={sorted(FULL_CONFIG['tier_rollup'].keys())}."
)

# ============================================================
# SECTION 5: Load the real Gate 5 decision-report CSV. keep_default_na=False / na_values=[] is a
# deliberate, documented choice: Gate 5's own code writes reason_codes/reason_evidence as an
# empty string "" (never null) for a NONE-tier cluster ("|".join([]) == ""), but pandas' DEFAULT
# NA-string heuristic treats a literal empty CSV field as NaN on read - a mis-parse relative to
# what was actually written, not a real null. Reading with the NA heuristic disabled is the
# byte-accurate re-read of the real file, not an invented substitute.
# ============================================================
SOURCE_CSV_PATH = ARTIFACTS_DIR / "gate5_cluster_decision_report.csv"
assert SOURCE_CSV_PATH.exists(), f"{SOURCE_CSV_PATH} missing - run BP4 Gate 5 for real first."

with open(SOURCE_CSV_PATH, "rb") as f:
    source_csv_bytes = f.read()
SOURCE_CSV_SHA256 = hashlib.sha256(source_csv_bytes).hexdigest()
print(
    f"[OK] Real source CSV read: {SOURCE_CSV_PATH.relative_to(PROJECT_ROOT)} "
    f"({len(source_csv_bytes):,} bytes, sha256={SOURCE_CSV_SHA256[:16]}...)"
)

raw_df = pd.read_csv(SOURCE_CSV_PATH, keep_default_na=False, na_values=[])
n_rows_loaded = len(raw_df)
assert n_rows_loaded == N_CLUSTERS_EXPECTED, (
    f"[CHECK FAILED] Loaded {n_rows_loaded:,} real rows but Gate 5's config block recorded "
    f"n_clusters_reported={N_CLUSTERS_EXPECTED:,} - the CSV on disk has drifted since Gate 5."
)
print(f"[OK] Real decision report loaded: {n_rows_loaded:,} rows (matches Gate 5's recorded count).")

EXPECTED_COLS = CLUSTER_KEY + [
    "n_complaints_total",
    "first_complaint_date",
    "last_complaint_date",
    "n_active_months",
    "avg_response_lag_days",
    "banking77_coverage_fraction",
    "is_recurring_cluster",
    "recurring_flag",
    "elevated_lag_flag",
    "high_volume_flag",
    "review_priority_score",
    "review_priority_tier",
    "reason_codes",
    "reason_evidence",
]
assert list(raw_df.columns) == EXPECTED_COLS, (
    f"[CHECK FAILED] Real column set/order does not match Gate 5's own documented output_cols: "
    f"{list(raw_df.columns)} != {EXPECTED_COLS}"
)
for barred in BARRED_JOURNEY_COLUMNS:
    assert barred not in raw_df.columns, f"[CHECK FAILED] barred column '{barred}' present."

n_dup_keys = int(raw_df.duplicated(subset=CLUSTER_KEY).sum())
assert n_dup_keys == 0, (
    f"[CHECK FAILED] {n_dup_keys} duplicate real CLUSTER_KEY rows found - the issue-cluster key "
    "is not actually unique in this real file, so it cannot safely serve as a lookup index."
)
print(
    f"[OK] Real CLUSTER_KEY {CLUSTER_KEY} verified unique across all {n_rows_loaded:,} real rows "
    "- BP4's only real, disclosed identifier (no customer/consumer identifier exists in this "
    "extract - Gate 1's own live-verified finding), and this notebook's physical sort/index key."
)

# ============================================================
# SECTION 6: Build the typed, sorted, indexed frame. Booleans are cast from real "True"/"False"
# text to a real bool dtype; the two real date columns are cast from real "YYYY-MM-DD" text to a
# real typed Date column (both ISO-unambiguous, live-verified below to round-trip exactly) so a
# downstream query engine gets typed columns, not opaque strings. The frame is then sorted by the
# real CLUSTER_KEY (WARP: one vectorized Polars sort, no per-row Python loop) - the natural
# physical index for the lookup service's real query pattern (point lookup by a real issue-cluster
# key, and prefix scan by a real Company).
# ============================================================
BOOL_COLS = ["is_recurring_cluster", "recurring_flag", "elevated_lag_flag", "high_volume_flag"]
for c in BOOL_COLS:
    # pandas' C parser already infers a native bool dtype for a "True"/"False"-only text column
    # (live-confirmed against this real file) - cast defensively rather than assumed, since a
    # future pandas/engine change silently falling back to strings must not be a silent bug here.
    real_values = set(pd.Series(raw_df[c]).astype(str).unique())
    assert real_values.issubset(
        {"True", "False"}
    ), f"[CHECK FAILED] Unexpected real value(s) in boolean column '{c}': {real_values}"

typed_df = raw_df.copy()
for c in BOOL_COLS:
    typed_df[c] = typed_df[c].astype(str).map({"True": True, "False": False}).astype(bool)

indexed_frame = pl.from_pandas(typed_df).with_columns(
    [
        pl.col("first_complaint_date").str.strptime(pl.Date, "%Y-%m-%d"),
        pl.col("last_complaint_date").str.strptime(pl.Date, "%Y-%m-%d"),
    ]
)
indexed_frame = indexed_frame.sort(CLUSTER_KEY)
print(
    f"[OK] Typed, sorted index frame built: {indexed_frame.height:,} rows, "
    f"{len(indexed_frame.columns)} columns, sorted by real CLUSTER_KEY {CLUSTER_KEY}."
)

# ============================================================
# SECTION 7: Write the Parquet artifact (idempotent overwrite-in-place). 37,160 real rows /
# ~370KB fits comfortably in one row group at this project's WARP RAM ceiling with no physical
# partitioning needed - the CLUSTER_KEY sort order alone is this artifact's real "index" (a
# predicate/range scan over a sorted column, standard Parquet/Polars practice at this real row
# count; multi-file partitioning would only add overhead here, never help, at 37K real rows).
# ============================================================
PARQUET_PATH = MODELS_DIR / "bp4_decision_artifact_index.parquet"
indexed_frame.write_parquet(PARQUET_PATH)
with open(PARQUET_PATH, "rb") as f:
    parquet_bytes_run1 = f.read()
parquet_sha256_run1 = hashlib.sha256(parquet_bytes_run1).hexdigest()
print(
    f"[SAVED] {PARQUET_PATH.relative_to(PROJECT_ROOT)} "
    f"({len(parquet_bytes_run1):,} bytes, sha256={parquet_sha256_run1[:16]}...)"
)

# ============================================================
# SECTION 8: Verification - (a) idempotency: re-write the same artifact in this same run and
# assert byte-identical output, proving the "idempotent overwrite" claim rather than merely
# asserting it; (b) round-trip fidelity: reload the Parquet and compare EVERY real value against
# the raw CSV read, across the FULL real population (37,160 rows - small enough that this project's
# own "full population, not a sample" precedent, already used at Gate 5, applies here too rather
# than settling for a partial sample check).
# ============================================================
indexed_frame.write_parquet(PARQUET_PATH)
with open(PARQUET_PATH, "rb") as f:
    parquet_bytes_run2 = f.read()
parquet_sha256_run2 = hashlib.sha256(parquet_bytes_run2).hexdigest()
idempotent_rewrite_matches = parquet_sha256_run1 == parquet_sha256_run2
print(
    f"[CHECK] In-run idempotent re-write: run1 sha256={parquet_sha256_run1[:16]}..., "
    f"run2 sha256={parquet_sha256_run2[:16]}... -> "
    f"{'IDENTICAL' if idempotent_rewrite_matches else 'DIFFERENT'}"
)
PARQUET_SHA256 = parquet_sha256_run2

reloaded = pl.read_parquet(PARQUET_PATH)
assert reloaded.height == n_rows_loaded, "[CHECK FAILED] row count changed after Parquet reload."

reloaded_pd = reloaded.to_pandas()
reloaded_pd["first_complaint_date"] = reloaded_pd["first_complaint_date"].astype(str)
reloaded_pd["last_complaint_date"] = reloaded_pd["last_complaint_date"].astype(str)

raw_sorted = raw_df.sort_values(CLUSTER_KEY).reset_index(drop=True)
reloaded_sorted = reloaded_pd.sort_values(CLUSTER_KEY).reset_index(drop=True)

NUMERIC_COLS = {
    "n_complaints_total",
    "n_active_months",
    "avg_response_lag_days",
    "banking77_coverage_fraction",
    "review_priority_score",
}
compare_cols = [c for c in EXPECTED_COLS if c not in BOOL_COLS]
n_value_mismatches = 0
for c in compare_cols:
    if c in NUMERIC_COLS:
        col_eq = (
            pd.to_numeric(raw_sorted[c]).astype(float) - pd.to_numeric(reloaded_sorted[c]).astype(float)
        ).abs() < 1e-9
    else:
        col_eq = raw_sorted[c].astype(str) == reloaded_sorted[c].astype(str)
    n_value_mismatches += int((~col_eq).sum())
for c in BOOL_COLS:
    col_eq = raw_sorted[c].astype(str).map({"True": True, "False": False}) == reloaded_sorted[c]
    n_value_mismatches += int((~col_eq).sum())

print(
    f"[CHECK] Full value-for-value round-trip comparison across all {n_rows_loaded:,} real rows x "
    f"{len(EXPECTED_COLS)} columns (CSV vs reloaded Parquet): {n_value_mismatches} mismatches."
)
assert n_value_mismatches == 0, f"[CHECK FAILED] {n_value_mismatches} CSV-vs-Parquet value mismatches."

# ============================================================
# SECTION 9: Write a human-readable metadata sidecar (mirrors BP1/BP2's own
# bp{1,2}_model_metadata.json pattern, adapted: no accuracy fields - this artifact is a persisted,
# indexed REPORTING record set, never a fitted model).
# ============================================================
generated_at = datetime.now(timezone.utc).isoformat()
TIER_COUNTS = {tier: block["n_clusters"] for tier, block in FULL_CONFIG["tier_rollup"].items()}

metadata_record = {
    "bp_id": "bp4",
    "artifact_type": "decision_records_index",
    "artifact_description": (
        "Fast-loadable, typed, CLUSTER_KEY-sorted Parquet index of BP4 Gate 5's real per-issue-"
        "cluster review-priority decision/reporting records - not a trained model (BP4 fits none)."
    ),
    "source_csv_relative_path": str(SOURCE_CSV_PATH.relative_to(PROJECT_ROOT)),
    "source_csv_sha256": SOURCE_CSV_SHA256,
    "source_csv_size_bytes": len(source_csv_bytes),
    "parquet_relative_path": str(PARQUET_PATH.relative_to(PROJECT_ROOT)),
    "parquet_sha256": PARQUET_SHA256,
    "parquet_size_bytes": len(parquet_bytes_run2),
    "n_rows": n_rows_loaded,
    "n_columns": len(EXPECTED_COLS),
    "sort_key": CLUSTER_KEY,
    "tier_counts": TIER_COUNTS,
    "idempotent_rewrite_verified": idempotent_rewrite_matches,
    "round_trip_value_mismatches": n_value_mismatches,
    "round_trip_verified": n_value_mismatches == 0,
    "generated_at_utc": generated_at,
}
METADATA_PATH = MODELS_DIR / "bp4_decision_artifact_metadata.json"
with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata_record, f, indent=2)
print(f"[SAVED] {METADATA_PATH.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 10: Note on model_inventory_entry.json - BP4 Gate 6's own governance summary
# (notebooks/bp4_customer_journey_analytics/artifacts/gate6_governance_summary.json) already
# states model_inventory_applicability: "NOT_APPLICABLE - BP4 registers no trained model". No such
# inventory file exists for BP4 (confirmed live below), and none is fabricated here to force a
# parallel with BP1/BP2/BP3's own model_inventory_entry.json - the metadata sidecar (Section 9)
# and this gate block (Section 11) together ARE this BP's honest governance record for what it
# actually persists.
# ============================================================
bp4_model_inventory_path = ARTIFACTS_DIR / "model_inventory_entry.json"
bp4_has_model_inventory = bp4_model_inventory_path.exists()
print(
    f"[OK] BP4 model_inventory_entry.json presence check: {bp4_has_model_inventory} (expected "
    "False - BP4 registers no trained model, per Gate 6's own already-recorded finding)."
)

# ============================================================
# SECTION 11: Write the Decision Artifact Persistence config block (marker-based, order-
# independent - reuses bp1_config_sync.py unmodified, same helper every gate/step in this project
# uses).
# ============================================================
persistence_marker = (
    "# --- Decision Artifact Persistence (Hardening Step 2) results (appended, idempotent overwrite) ---"
)
persistence_block_lines = [
    "decision_artifact_persistence:",
    f'  parquet_relative_path: "{PARQUET_PATH.relative_to(PROJECT_ROOT).as_posix()}"',
    f'  parquet_sha256: "{PARQUET_SHA256}"',
    f'  source_csv_sha256: "{SOURCE_CSV_SHA256}"',
    f"  n_rows: {n_rows_loaded}",
    f"  n_columns: {len(EXPECTED_COLS)}",
    f"  sort_key: {CLUSTER_KEY}",
    f"  idempotent_rewrite_verified: {idempotent_rewrite_matches}",
    f"  round_trip_verified: {n_value_mismatches == 0}",
    f'  generated_at_utc: "{generated_at}"',
]
write_gate_block(BP4_CONFIG_PATH, persistence_marker, persistence_block_lines)
print(f"[SAVED] {BP4_CONFIG_PATH.relative_to(PROJECT_ROOT)} (decision_artifact_persistence block)")

with open(BP4_CONFIG_PATH, "r", encoding="utf-8") as f:
    _post_write_config_text = f.read()
config_block_actually_written = persistence_marker in _post_write_config_text

# ============================================================
# SECTION 12: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
checks = {
    "gate5_prerequisite_confirmed": gate5_confirmed,
    "source_csv_row_count_matches_gate5_config": n_rows_loaded == N_CLUSTERS_EXPECTED,
    "real_columns_match_gate5_output_cols": list(raw_df.columns) == EXPECTED_COLS,
    "no_barred_column_present": all(b not in raw_df.columns for b in BARRED_JOURNEY_COLUMNS),
    "cluster_key_unique_in_source": n_dup_keys == 0,
    "parquet_file_written": PARQUET_PATH.exists(),
    "parquet_file_nonempty": len(parquet_bytes_run2) > 0,
    "parquet_reload_row_count_matches_source": reloaded.height == n_rows_loaded,
    "parquet_full_round_trip_value_match": n_value_mismatches == 0,
    "parquet_idempotent_rewrite_byte_identical": idempotent_rewrite_matches,
    "metadata_sidecar_written": METADATA_PATH.exists(),
    "no_model_inventory_fabricated_for_bp4": not bp4_has_model_inventory,
    "config_gate_block_written": config_block_actually_written,
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print(
    f"\n[ALL CHECKS PASSED] BP4 decision-artifact persistence complete. {n_rows_loaded:,} real "
    f"per-cluster decision records persisted to "
    f"{PARQUET_PATH.relative_to(PROJECT_ROOT)} ({len(parquet_bytes_run2):,} bytes), sorted by real "
    f"CLUSTER_KEY {CLUSTER_KEY}. Round-trip verified value-for-value across the full real "
    f"population with 0 mismatches; idempotent re-write verified byte-identical. Tier counts: "
    + ", ".join(f"{tier}={count:,}" for tier, count in sorted(TIER_COUNTS.items()))
    + ". No trained model persisted (BP4 fits none) and no model_inventory_entry.json fabricated. "
    "Ready for Hardening Step 3 (read-only lookup/query FastAPI service)."
)
